In [1]:
import pandas as pd
import nltk
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

In [2]:
# Download corpus stopwords dan tokenizer dari NLTK
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab') # <-- Tambahin baris ini

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\LENOVO\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\LENOVO\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\LENOVO\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [3]:
print("✅ Setup Jupyter dan Library NLP Berhasil!")

✅ Setup Jupyter dan Library NLP Berhasil!


In [4]:
import re
import pandas as pd
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

# 1. Inisialisasi Stemmer (Sastrawi)
factory = StemmerFactory()
stemmer = factory.create_stemmer()

# 2. Setup Stopwords (JAGA KATA NEGASI!)
stop_words = set(stopwords.words('indonesian'))
# Tambahan kata negasi alay biar aman
negasi = {'tidak', 'bukan', 'tak', 'jangan', 'belum', 'tanpa', 'enggak', 'ga', 'gak', 'ngga', 'nggak'}
stop_words = stop_words - negasi

# 3. Load Kamus Slang yang udah kita bikin tadi
# Perhatikan path-nya '../' karena posisi notebook ada di dalam folder 'notebooks'
slang_df = pd.read_csv('../data/dictionaries/slang_dictionary.csv')
slang_dict = dict(zip(slang_df['slang'], slang_df['formal']))

def normalize_text(text):
    """Fungsi untuk mengubah kata alay/slang menjadi kata baku"""
    words = text.split()
    normalized_words = [slang_dict.get(word, word) for word in words]
    return ' '.join(normalized_words)

def preprocess_text(text):
    """Pipeline utama pembersihan teks"""
    if pd.isna(text):
        return ""
    
    text = str(text)
    
    # a. Cleaning (Hapus URL, mention, hashtag, dan angka)
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'\@\w+|\#\w+', '', text)
    text = re.sub(r'\d+', '', text)
    
    # b. Hapus tanda baca (diganti spasi biar kata tidak nempel)
    text = re.sub(r'[^\w\s]', ' ', text)
    
    # c. Case Folding (huruf kecil & hapus spasi berlebih)
    text = text.lower().strip()
    text = re.sub(r'\s+', ' ', text)
    
    # d. Normalisasi (Slang)
    text = normalize_text(text)
    
    # e. Tokenisasi & Stopword Removal
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in stop_words]
    
    # f. Stemming (Kembalikan ke kata dasar)
    tokens = [stemmer.stem(t) for t in tokens]
    
    return ' '.join(tokens)

# Mari kita tes!
contoh_teks = "Aplikasinya jelek bgt, gak bisa login!!! Sering error dan nyusahin pdhl blm update https://test.com"
hasil = preprocess_text(contoh_teks)

print("Teks Asli   :", contoh_teks)
print("Hasil Clean :", hasil)

Teks Asli   : Aplikasinya jelek bgt, gak bisa login!!! Sering error dan nyusahin pdhl blm update https://test.com
Hasil Clean : aplikasi jelek banget tidak login error susah belum baru


In [5]:
import pandas as pd

# 1. Load Lexicon
pos_lex = pd.read_csv('../data/dictionaries/inset_positive.csv')
neg_lex = pd.read_csv('../data/dictionaries/inset_negative.csv')

# Perbaikan 1: Pangkas spasi tersembunyi (strip) dan abaikan data kosong (NaN)
pos_dict = {str(word).strip(): int(weight) for word, weight in zip(pos_lex['word'], pos_lex['weight']) if pd.notna(word)}
neg_dict = {str(word).strip(): int(weight) for word, weight in zip(neg_lex['word'], neg_lex['weight']) if pd.notna(word)}

# 2. Amunisi Aturan Khusus
kata_negasi = {'tidak', 'bukan', 'tak', 'jangan', 'belum', 'tanpa', 'enggak', 'ga', 'gak'}
# Perbaikan 2: Daftar kata subjek yang harus netral (diabaikan skornya)
kata_abaikan = {'aplikasi', 'app', 'apk', 'shopee', 'tokopedia', 'blibli'}

def label_sentiment_debug(clean_text):
    """
    Fungsi pelabelan lengkap dengan fitur "X-Ray" (Debugging) 
    untuk melihat darimana skor didapatkan.
    """
    if pd.isna(clean_text) or clean_text == "":
        return 'Netral'
        
    tokens = clean_text.split()
    score = 0
    
    print(f"🔍 Membedah kalimat: '{clean_text}'")
    
    for i, token in enumerate(tokens):
        # Abaikan subjek aplikasi
        if token in kata_abaikan:
            print(f"   -> '{token}': Diabaikan (Subjek E-commerce)")
            continue
            
        word_score = 0
        
        # Cek ke kamus
        if token in pos_dict:
            word_score = pos_dict[token]
            asal = "Kamus Positif"
        elif token in neg_dict:
            word_score = neg_dict[token]
            asal = "Kamus Negatif"
        else:
            print(f"   -> '{token}': 0 (Tidak ada di InSet)")
            continue
            
        # Logika Pintar Negasi
        if i > 0 and tokens[i-1] in kata_negasi:
            word_score = word_score * -1
            print(f"   -> '{token}': {word_score} (Skor dibalik karena '{tokens[i-1]}')")
        else:
            print(f"   -> '{token}': {word_score} ({asal})")
            
        score += word_score
        
    if score > 0:
        label = 'Positif'
    elif score < 0:
        label = 'Negatif'
    else:
        label = 'Netral'
        
    print(f"🎯 Total Skor: {score} => Label: {label}\n")
    return label

# ==========================================
# TEST DEBUGGER
# ==========================================
_ = label_sentiment_debug("aplikasi bagus")
_ = label_sentiment_debug("aplikasi tidak bagus")

🔍 Membedah kalimat: 'aplikasi bagus'
   -> 'aplikasi': Diabaikan (Subjek E-commerce)
   -> 'bagus': 2 (Kamus Positif)
🎯 Total Skor: 2 => Label: Positif

🔍 Membedah kalimat: 'aplikasi tidak bagus'
   -> 'aplikasi': Diabaikan (Subjek E-commerce)
   -> 'tidak': -5 (Kamus Negatif)
   -> 'bagus': -2 (Skor dibalik karena 'tidak')
🎯 Total Skor: -7 => Label: Negatif



In [8]:
import os
import pandas as pd

# ==========================================
# 1. RADAR LOKASI FOLDER
# ==========================================
if os.path.exists('../data/raw'):
    base_path = '../'
elif os.path.exists('data/raw'): 
    base_path = './'
else:
    raise FileNotFoundError("Waduh, foldernya tetep gak ketemu nih.")

os.makedirs(os.path.join(base_path, 'data', 'processed'), exist_ok=True)
print(f"📍 Radar aktif! Data ditemukan di jalur: {base_path}data/raw/")

# ==========================================
# 2. FUNGSI PELABELAN FINAL
# ==========================================
def label_sentiment_final(clean_text):
    if pd.isna(clean_text) or clean_text == "":
        return 'Netral'
        
    tokens = str(clean_text).split()
    score = 0
    
    for i, token in enumerate(tokens):
        if token in kata_abaikan: continue 
            
        word_score = 0
        if token in pos_dict: word_score = pos_dict[token]
        elif token in neg_dict: word_score = neg_dict[token]
            
        if i > 0 and tokens[i-1] in kata_negasi:
            word_score = word_score * -1 
            
        score += word_score
        
    if score > 0: return 'Positif'
    elif score < 0: return 'Negatif'
    else: return 'Netral'

# ==========================================
# 3. PROSES EKSEKUSI MASSAL (REVISI TARGET KOLOM)
# ==========================================
file_names = ['raw_shopee.csv', 'raw_tokopedia.csv', 'raw_blibli.csv']

for file in file_names:
    print(f"\n🚀 Memulai proses untuk: {file}")
    file_path = os.path.join(base_path, 'data', 'raw', file)
    
    if not os.path.exists(file_path):
        print(f"   ❌ File {file} tidak ditemukan!")
        continue
        
    try:
        df = pd.read_csv(file_path)
        
        # 🎯 PERBAIKAN: Kita tembak langsung kolom 'text' sesuai hasil olah TKP!
        col_text = 'text'
        
        print(f"   ⏳ 1. Sastrawi sedang membersihkan teks ulasan... (Tunggu ya, butuh waktu ☕)")
        df['clean_text'] = df[col_text].apply(preprocess_text)
        
        df = df.dropna(subset=['clean_text'])
        df = df[df['clean_text'].str.strip() != '']
        
        print(f"   ⏳ 2. InSet Lexicon sedang memberikan label...")
        df['label'] = df['clean_text'].apply(label_sentiment_final)
        
        save_name = file.replace('raw_', 'clean_')
        save_path = os.path.join(base_path, 'data', 'processed', save_name)
        df.to_csv(save_path, index=False)
        
        print(f"   ✅ Selesai! Data berhasil disimpan di: {save_path}")
        print(f"   📊 Distribusi Label yang Sebenarnya:\n{df['label'].value_counts().to_string()}")
        
    except Exception as e:
        print(f"   ❌ Error saat memproses {file}: {e}")

print("\n🎉 BOOYAH YANG ASLI! SEMUA DATA SELESAI DI-PREPROCESS & DILABELI!")

📍 Radar aktif! Data ditemukan di jalur: ../data/raw/

🚀 Memulai proses untuk: raw_shopee.csv
   ⏳ 1. Sastrawi sedang membersihkan teks ulasan... (Tunggu ya, butuh waktu ☕)
   ⏳ 2. InSet Lexicon sedang memberikan label...
   ✅ Selesai! Data berhasil disimpan di: ../data\processed\clean_shopee.csv
   📊 Distribusi Label yang Sebenarnya:
label
Positif    6382
Negatif    3269
Netral      340

🚀 Memulai proses untuk: raw_tokopedia.csv
   ⏳ 1. Sastrawi sedang membersihkan teks ulasan... (Tunggu ya, butuh waktu ☕)
   ⏳ 2. InSet Lexicon sedang memberikan label...
   ✅ Selesai! Data berhasil disimpan di: ../data\processed\clean_tokopedia.csv
   📊 Distribusi Label yang Sebenarnya:
label
Positif    5753
Negatif    3950
Netral      295

🚀 Memulai proses untuk: raw_blibli.csv
   ⏳ 1. Sastrawi sedang membersihkan teks ulasan... (Tunggu ya, butuh waktu ☕)
   ⏳ 2. InSet Lexicon sedang memberikan label...
   ✅ Selesai! Data berhasil disimpan di: ../data\processed\clean_blibli.csv
   📊 Distribusi Label y

In [2]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

print("🚀 MEMULAI FASE MACHINE LEARNING: EKSTRAKSI FITUR (TF-IDF)")

# 1. Load salah satu data yang sudah bersih (contoh: Shopee)
df_ml = pd.read_csv('../data/processed/clean_shopee.csv')

# 2. Ekstraksi Fitur dengan TF-IDF 
# (Kita batasi 5000 kata paling sering muncul biar RAM laptop lo nggak meledak)
tfidf_vectorizer = TfidfVectorizer(max_features=5000)

# Menerapkan TF-IDF ke kolom clean_text
X = tfidf_vectorizer.fit_transform(df_ml['clean_text'])
y = df_ml['label'] # Target yang mau ditebak

# 3. Train-Test Split (Bagi data 80% untuk belajar, 20% untuk ujian)
# JANGAN LUPA: stratify=y biar pembagian Positif/Negatif/Netralnya seimbang!
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

print("✅ Data berhasil diubah jadi angka (Vectorized)!")
print(f"📚 Jumlah data untuk Training (Belajar): {X_train.shape[0]} ulasan")
print(f"📝 Jumlah data untuk Testing (Ujian): {X_test.shape[0]} ulasan")
print(f"🧠 Jumlah fitur (kata unik) yang dipelajari: {X_train.shape[1]} kata")

🚀 MEMULAI FASE MACHINE LEARNING: EKSTRAKSI FITUR (TF-IDF)
✅ Data berhasil diubah jadi angka (Vectorized)!
📚 Jumlah data untuk Training (Belajar): 7992 ulasan
📝 Jumlah data untuk Testing (Ujian): 1999 ulasan
🧠 Jumlah fitur (kata unik) yang dipelajari: 5000 kata


In [3]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.metrics import classification_report, accuracy_score

print("🤖 MEMULAI TRAINING MODEL MACHINE LEARNING...\n")

# 1. Inisialisasi Model
nb_model = MultinomialNB()

# Pake n_estimators=100 (100 pohon). n_jobs=-1 biar pake seluruh core CPU laptop lo!
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1) 

# 2. Bikin Ensemble Model (Soft Voting)
ensemble_model = VotingClassifier(
    estimators=[('Naive Bayes', nb_model), ('Random Forest', rf_model)],
    voting='soft'
)

# 3. Training Waktu! (Menyuruh mesin belajar dari 7.992 ulasan)
print("⏳ Melatih Naive Bayes (Biasanya secepat kilat)...")
nb_model.fit(X_train, y_train)

print("⏳ Melatih Random Forest (Sabar, pohonnya lagi ditanam... 🌳)")
rf_model.fit(X_train, y_train)

print("⏳ Melatih Ensemble Model (Menggabungkan dua otak)...")
ensemble_model.fit(X_train, y_train)

# 4. Fase Ujian (Menyuruh mesin menebak 1.999 ulasan yang belum pernah dia lihat)
print("\n🎯 FASE UJIAN (PREDIKSI DATA TEST)...")
y_pred_nb = nb_model.predict(X_test)
y_pred_rf = rf_model.predict(X_test)
y_pred_ensemble = ensemble_model.predict(X_test)

# 5. Rapor Hasil Ujian
print("\n" + "="*50)
print("🏆 HASIL AKURASI MODEL (BASELINE LEXICON)")
print("="*50)
print(f"Akurasi Naive Bayes   : {accuracy_score(y_test, y_pred_nb):.4f}")
print(f"Akurasi Random Forest : {accuracy_score(y_test, y_pred_rf):.4f}")
print(f"Akurasi Ensemble      : {accuracy_score(y_test, y_pred_ensemble):.4f}")
print("="*50)

# Munculin raport detail buat Ensemble
print("\n📊 Rapor Detail Ensemble Model:")
print(classification_report(y_test, y_pred_ensemble))

🤖 MEMULAI TRAINING MODEL MACHINE LEARNING...

⏳ Melatih Naive Bayes (Biasanya secepat kilat)...
⏳ Melatih Random Forest (Sabar, pohonnya lagi ditanam... 🌳)
⏳ Melatih Ensemble Model (Menggabungkan dua otak)...

🎯 FASE UJIAN (PREDIKSI DATA TEST)...

🏆 HASIL AKURASI MODEL (BASELINE LEXICON)
Akurasi Naive Bayes   : 0.7294
Akurasi Random Forest : 0.8069
Akurasi Ensemble      : 0.7779

📊 Rapor Detail Ensemble Model:
              precision    recall  f1-score   support

     Negatif       0.85      0.50      0.63       654
      Netral       0.00      0.00      0.00        68
     Positif       0.76      0.96      0.85      1277

    accuracy                           0.78      1999
   macro avg       0.54      0.49      0.49      1999
weighted avg       0.77      0.78      0.75      1999



\\wsl$\Ubuntu-22.04\home\shandy28\sentiment-pipeline\.venv-1\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
\\wsl$\Ubuntu-22.04\home\shandy28\sentiment-pipeline\.venv-1\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
\\wsl$\Ubuntu-22.04\home\shandy28\sentiment-pipeline\.venv-1\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavio

In [4]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.metrics import classification_report, accuracy_score

print("🤖 MEMULAI TRAINING MODEL MACHINE LEARNING...\n")

# 1. Inisialisasi Model
nb_model = MultinomialNB()

# Pake n_estimators=100 (100 pohon). n_jobs=-1 biar pake seluruh core CPU laptop lo!
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1) 

# 2. Bikin Ensemble Model (Soft Voting)
ensemble_model = VotingClassifier(
    estimators=[('Naive Bayes', nb_model), ('Random Forest', rf_model)],
    voting='soft'
)

# 3. Training Waktu! (Menyuruh mesin belajar dari 7.992 ulasan)
print("⏳ Melatih Naive Bayes (Biasanya secepat kilat)...")
nb_model.fit(X_train, y_train)

print("⏳ Melatih Random Forest (Sabar, pohonnya lagi ditanam... 🌳)")
rf_model.fit(X_train, y_train)

print("⏳ Melatih Ensemble Model (Menggabungkan dua otak)...")
ensemble_model.fit(X_train, y_train)

# 4. Fase Ujian (Menyuruh mesin menebak 1.999 ulasan yang belum pernah dia lihat)
print("\n🎯 FASE UJIAN (PREDIKSI DATA TEST)...")
y_pred_nb = nb_model.predict(X_test)
y_pred_rf = rf_model.predict(X_test)
y_pred_ensemble = ensemble_model.predict(X_test)

# 5. Rapor Hasil Ujian
print("\n" + "="*50)
print("🏆 HASIL AKURASI MODEL (BASELINE LEXICON)")
print("="*50)
print(f"Akurasi Naive Bayes   : {accuracy_score(y_test, y_pred_nb):.4f}")
print(f"Akurasi Random Forest : {accuracy_score(y_test, y_pred_rf):.4f}")
print(f"Akurasi Ensemble      : {accuracy_score(y_test, y_pred_ensemble):.4f}")
print("="*50)

# Munculin raport detail buat Ensemble
print("\n📊 Rapor Detail Ensemble Model:")
print(classification_report(y_test, y_pred_ensemble))

🤖 MEMULAI TRAINING MODEL MACHINE LEARNING...

⏳ Melatih Naive Bayes (Biasanya secepat kilat)...
⏳ Melatih Random Forest (Sabar, pohonnya lagi ditanam... 🌳)
⏳ Melatih Ensemble Model (Menggabungkan dua otak)...

🎯 FASE UJIAN (PREDIKSI DATA TEST)...

🏆 HASIL AKURASI MODEL (BASELINE LEXICON)
Akurasi Naive Bayes   : 0.7294
Akurasi Random Forest : 0.8069
Akurasi Ensemble      : 0.7779

📊 Rapor Detail Ensemble Model:
              precision    recall  f1-score   support

     Negatif       0.85      0.50      0.63       654
      Netral       0.00      0.00      0.00        68
     Positif       0.76      0.96      0.85      1277

    accuracy                           0.78      1999
   macro avg       0.54      0.49      0.49      1999
weighted avg       0.77      0.78      0.75      1999



\\wsl$\Ubuntu-22.04\home\shandy28\sentiment-pipeline\.venv-1\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
\\wsl$\Ubuntu-22.04\home\shandy28\sentiment-pipeline\.venv-1\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
\\wsl$\Ubuntu-22.04\home\shandy28\sentiment-pipeline\.venv-1\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavio

In [5]:
from imblearn.over_sampling import SMOTE
from collections import Counter
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.metrics import classification_report, accuracy_score

print("🛠️ MEMULAI OPERASI SMOTE (PENYEMBUHAN DATA JOMPLANG)...\n")

# 1. Cek Populasi Sebelum SMOTE
print("📊 Populasi Kelas SEBELUM SMOTE (Data Belajar):")
print(Counter(y_train))

# 2. Terapkan keajaiban SMOTE
# random_state=42 agar hasilnya konsisten kalau di-run berkali-kali
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

# 3. Cek Populasi Sesudah SMOTE
print("\n📈 Populasi Kelas SESUDAH SMOTE (Data Belajar):")
print(Counter(y_train_smote))
print("\n(Bisa dilihat sekarang jumlahnya udah rata semua!)\n")

# ==========================================
# RETRAINING MODEL DENGAN DATA BARU
# ==========================================
print("🤖 MELATIH ULANG MODEL DENGAN DATA SEIMBANG...")

# Inisialisasi ulang model biar ingatannya bersih
nb_model_smote = MultinomialNB()
rf_model_smote = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
ensemble_model_smote = VotingClassifier(
    estimators=[('Naive Bayes', nb_model_smote), ('Random Forest', rf_model_smote)],
    voting='soft'
)

print("   ⏳ Melatih Naive Bayes...")
nb_model_smote.fit(X_train_smote, y_train_smote)

print("   ⏳ Melatih Random Forest (Nanam pohon lagi... 🌳)")
rf_model_smote.fit(X_train_smote, y_train_smote)

print("   ⏳ Melatih Ensemble Model...")
ensemble_model_smote.fit(X_train_smote, y_train_smote)

# ==========================================
# UJIAN ULANG
# ==========================================
print("\n🎯 FASE UJIAN KEDUA (PREDIKSI DATA TEST)...")
y_pred_nb_smote = nb_model_smote.predict(X_test)
y_pred_rf_smote = rf_model_smote.predict(X_test)
y_pred_ensemble_smote = ensemble_model_smote.predict(X_test)

# Raport Baru
print("\n" + "="*50)
print("🏆 HASIL AKURASI MODEL (SETELAH SMOTE)")
print("="*50)
print(f"Akurasi Naive Bayes   : {accuracy_score(y_test, y_pred_nb_smote):.4f}")
print(f"Akurasi Random Forest : {accuracy_score(y_test, y_pred_rf_smote):.4f}")
print(f"Akurasi Ensemble      : {accuracy_score(y_test, y_pred_ensemble_smote):.4f}")
print("="*50)

print("\n📊 Rapor Detail Ensemble Model (SMOTE):")
print(classification_report(y_test, y_pred_ensemble_smote))

🛠️ MEMULAI OPERASI SMOTE (PENYEMBUHAN DATA JOMPLANG)...

📊 Populasi Kelas SEBELUM SMOTE (Data Belajar):
Counter({'Positif': 5105, 'Negatif': 2615, 'Netral': 272})

📈 Populasi Kelas SESUDAH SMOTE (Data Belajar):
Counter({'Positif': 5105, 'Netral': 5105, 'Negatif': 5105})

(Bisa dilihat sekarang jumlahnya udah rata semua!)

🤖 MELATIH ULANG MODEL DENGAN DATA SEIMBANG...
   ⏳ Melatih Naive Bayes...
   ⏳ Melatih Random Forest (Nanam pohon lagi... 🌳)
   ⏳ Melatih Ensemble Model...

🎯 FASE UJIAN KEDUA (PREDIKSI DATA TEST)...

🏆 HASIL AKURASI MODEL (SETELAH SMOTE)
Akurasi Naive Bayes   : 0.6508
Akurasi Random Forest : 0.7969
Akurasi Ensemble      : 0.7794

📊 Rapor Detail Ensemble Model (SMOTE):
              precision    recall  f1-score   support

     Negatif       0.68      0.76      0.71       654
      Netral       0.04      0.01      0.02        68
     Positif       0.85      0.83      0.84      1277

    accuracy                           0.78      1999
   macro avg       0.52      0.5

In [7]:
import pandas as pd

# Mari kita intip hasil file yang baru aja di-generate
df_cek = pd.read_csv('../data/processed/clean_shopee.csv')

print("🔍 DETEKTIF DATA AKTIF!")
print("Daftar Kolom Asli:", df_cek.columns.tolist())
print("-" * 50)
print("Isi 5 Baris Pertama dari Kolom 'clean_text':")
print(df_cek['clean_text'].head())

🔍 DETEKTIF DATA AKTIF!
Daftar Kolom Asli: ['app', 'text', 'rating', 'date', 'scraped_at', 'clean_text', 'label']
--------------------------------------------------
Isi 5 Baris Pertama dari Kolom 'clean_text':
0    shopee
1    shopee
2    shopee
3    shopee
4    shopee
Name: clean_text, dtype: str


In [1]:
import pandas as pd

# Load data Blibli yang udah bersih
df_blibli = pd.read_csv('../data/processed/clean_blibli.csv')

# Filter khusus yang labelnya 'Positif'
df_positif = df_blibli[df_blibli['label'] == 'Positif']

print("🔍 BUKTI DATA POSITIF BLIBLI AMAN:")
# Tampilkan 10 baris pertama khusus yang positif
print(df_positif[['clean_text', 'label']].head(10))

🔍 BUKTI DATA POSITIF BLIBLI AMAN:
                                           clean_text    label
0   kirim lelet tidak kirim prioritas barang mahal...  Positif
1   beli barang paket antar alamat salah kurir tid...  Positif
2   ya barang jual asli transaksi click collect to...  Positif
5   kecewa belanja blibli niat cepat pakai go inst...  Positif
6   bagus sih beli milih expedisi jne daerah blita...  Positif
10  kirim nya lamaa banget sampe hari pelayanann n...  Positif
12  nanya expired produk yg beli nunggu antri meni...  Positif
14  bagus admin aplikasi upgrade keren tambahin fi...  Positif
15  hati olshop main modus tawar harga diskon gila...  Positif
16  jual langsung nonaktif indikasi langgar dll ka...  Positif
